In [15]:
import os
import numpy as np
import pandas as pd


def summarize_spectra_grouping_counts(
    metadata_path='/home/msp25gd/Downloads/res/meta/metadata.pkl',
    wave_min=None,
    wave_max=None,
    group_column=None,
    min_group_size=3,
    return_group_table=True,
):
    """
    Summarize early pipeline counts for spectra filtering and grouping.

    If wave_min/wave_max are not provided, this uses the same Ca II window
    logic used in metadata processing (line coverage for Ca II K and H).

    Returns:
    - summary dict with counts
    - optional group size table after wavelength filtering
    """
    metadata = pd.read_pickle(metadata_path).copy()

    # 1) Total spectra at the beginning: prefer raw fits_list count if available.
    total_spectra_source = 'metadata_rows'
    total_spectra_initial = int(len(metadata))
    default_fits_list_path = os.path.join(os.path.dirname(metadata_path), 'fits_list.npy')
    if os.path.exists(default_fits_list_path):
        try:
            total_spectra_initial = int(len(np.load(default_fits_list_path, allow_pickle=True)))
            total_spectra_source = default_fits_list_path
        except Exception:
            total_spectra_source = 'metadata_rows_fallback_load_error'

    # Resolve wavelength coverage columns.
    wmin_col = 'WAVELMIN' if 'WAVELMIN' in metadata.columns else ('wavelmin' if 'wavelmin' in metadata.columns else None)
    wmax_col = 'WAVELMAX' if 'WAVELMAX' in metadata.columns else ('wavelmax' if 'wavelmax' in metadata.columns else None)
    if wmin_col is None or wmax_col is None:
        raise KeyError('Could not find WAVELMIN/WAVELMAX columns in metadata.')

    window_source = 'explicit_args'
    if wave_min is None or wave_max is None:
        # Prefer pre-existing runtime variables if they were already defined upstream.
        g = globals()
        candidate_pairs = [
            ('line_min', 'line_max'),
            ('wave_min', 'wave_max'),
            ('ca_k', 'ca_h'),
            ('ca_h', 'ca_k'),
        ]
        resolved_pair = None
        for left_name, right_name in candidate_pairs:
            if left_name in g and right_name in g:
                left_val = g[left_name]
                right_val = g[right_name]
                try:
                    left_val = float(left_val)
                    right_val = float(right_val)
                except (TypeError, ValueError):
                    continue
                resolved_pair = (min(left_val, right_val), max(left_val, right_val))
                window_source = f'globals:{left_name}/{right_name}'
                break

        if resolved_pair is not None:
            wave_min, wave_max = resolved_pair
        else:
            # Fallback: reproduce the same constants/units from metadata.ipynb processing.
            ca_k_angstrom = 3933.66
            ca_h_angstrom = 3968.47
            if metadata[[wmin_col, wmax_col]].median().max() < 1000:
                ca_k = ca_k_angstrom / 10.0  # nm
                ca_h = ca_h_angstrom / 10.0  # nm
            else:
                ca_k = ca_k_angstrom         # Angstrom
                ca_h = ca_h_angstrom         # Angstrom
            wave_min = float(min(ca_k, ca_h))
            wave_max = float(max(ca_k, ca_h))
            window_source = 'processing_defaults_ca_doublet'
    else:
        wave_min = float(wave_min)
        wave_max = float(wave_max)

    # 2) Spectra that contain the stipulated wavelength range.
    covers_window = (metadata[wmin_col] <= wave_min) & (metadata[wmax_col] >= wave_max)
    metadata_in_window = metadata[covers_window].copy()
    spectra_in_window = int(len(metadata_in_window))

    # Resolve grouping column.
    if group_column is None:
        for candidate in ['New Groups', 'Reduced', 'reduced']:
            if candidate in metadata_in_window.columns:
                group_column = candidate
                break
    if group_column is None or group_column not in metadata_in_window.columns:
        raise KeyError("Could not resolve a grouping column. Pass group_column explicitly.")

    # Build group sizes after wavelength filtering.
    group_sizes = (
        metadata_in_window.groupby(group_column, dropna=False)
        .size()
        .reset_index(name='spectra_count')
        .sort_values('spectra_count', ascending=False)
        .reset_index(drop=True)
    )

    # 3) Total unique groups created.
    unique_groups_created = int(len(group_sizes))

    # 4) Groups that pass spectra > 2 filter (or configurable min_group_size).
    groups_pass_filter = int((group_sizes['spectra_count'] >= int(min_group_size)).sum())

    summary_text = (
        f"total number of spectra = {total_spectra_initial}\n"
        f"number of spectra inside the desired wavelength = {spectra_in_window}\n"
        f"number of unique groups = {unique_groups_created}\n"
        f"number of groups with {int(min_group_size)} or more spectra in it = {groups_pass_filter}"
    )

    summary = {
        'metadata_path': metadata_path,
        'total_spectra_source': total_spectra_source,
        'wavelength_window': [wave_min, wave_max],
        'wavelength_window_source': window_source,
        'group_column_used': group_column,
        'total_spectra_initial': total_spectra_initial,
        'spectra_in_wavelength_window': spectra_in_window,
        'unique_groups_created': unique_groups_created,
        'groups_with_spectra_ge_min_group_size': groups_pass_filter,
        'min_group_size': int(min_group_size),
        'summary_text': summary_text,
    }

    if return_group_table:
        return summary, group_sizes
    return summary


### Example usage (no manual wave_min/wave_max needed):
# summary, group_table = summarize_spectra_grouping_counts(min_group_size=3)
# print(summary['summary_text'])
# display(group_table.head(20))

In [16]:
# Example run: uses the processing-time Ca II window automatically.
summary, group_table = summarize_spectra_grouping_counts(
    min_group_size=3,
 )

print(summary['summary_text'])
display(group_table.head(20))

total number of spectra = 84188
number of spectra inside the desired wavelength = 32308
number of unique groups = 12827
number of groups with 3 or more spectra in it = 2887


,Reduced,spectra_count
0,speedymic,263
1,noname,236
2,scox1,189
3,betapictoris,171
4,wr11,155
5,rsoph,107
6,wasp7,89
7,grb,82
8,hd93129a,78
9,wr20a,75
